# silver_commission_reconciliation

Surfaces the drift between what the legacy VBA tool calculated and what the documented commission rules say it should have calculated. Reads directly from silver_commission_payouts where both values already live side by side.

This is the table a finance analyst would open to answer: "where did VBA get it wrong, by how much, and in which direction?" It's also the demo artifact that makes the business case for replacing the VBA tool — not just "the old process was bad" but here are 110 specific payouts, here are the agent names, here are the dollar amounts.

## Drift classification

Three patterns, each with a business interpretation:

- **PATTERN_A**: commission_rate recorded at wrong tier. Agent was underpaid. Root cause: VBA tier lookup had a one-step-down bug for non-BRONZE agents.
- **PATTERN_B**: program_modifier stored as 1.0 regardless of program. Agent was underpaid. Root cause: VBA program modifier lookup defaulted to 1.0 when the program code wasn't found in its internal table.
- **PATTERN_C**: volume bonus applied to non-qualifying agent. Agent was overpaid. Root cause: VBA volume bonus flag wasn't refreshed when agent's trailing premium dropped below threshold.

## Output

- `silver_citadel_mga.silver_commission_reconciliation`

One row per drifted payout. Columns include the payout details, both commission amounts, the variance (silver minus bronze), and the drift pattern classification.


In [2]:
from pyspark.sql.functions import (
    lit, current_timestamp, col, when, round as spark_round
)
import uuid
from datetime import datetime

SILVER_BATCH_ID = f"silver_{datetime.utcnow().strftime('%Y%m%d_%H%M%S')}_{uuid.uuid4().hex[:8]}"

# Source: silver_commission_payouts (already has both bronze and silver values).
SOURCE_TABLE = "silver_citadel_mga.dbo.silver_commission_payouts"
TARGET_TABLE = "silver_commission_reconciliation"

print(f"Silver batch ID: {SILVER_BATCH_ID}")
print(f"Source:          {SOURCE_TABLE}")
print(f"Target:          {TARGET_TABLE}")

StatementMeta(, 488ad106-7cea-435f-956d-2050f45c1443, 4, Finished, Available, Finished, False)

Silver batch ID: silver_20260603_031412_dfb09ab7
Source:          silver_citadel_mga.dbo.silver_commission_payouts
Target:          silver_commission_reconciliation


In [3]:
# Read silver_commission_payouts and filter to only drifted rows.
# A row has drift when bronze commission_amount != silver_commission_amount.

payouts_df = spark.read.table(SOURCE_TABLE)

drift_df = payouts_df.filter(
    col("commission_amount") != col("silver_commission_amount")
)

total_count = payouts_df.count()
drift_count = drift_df.count()

print(f"Total payouts:  {total_count}")
print(f"Drifted rows:   {drift_count} ({drift_count/total_count*100:.1f}%)")
print()
print(f"Columns available: {drift_df.columns}")

StatementMeta(, 488ad106-7cea-435f-956d-2050f45c1443, 5, Finished, Available, Finished, False)

Total payouts:  5000
Drifted rows:   110 (2.2%)

Columns available: ['payout_id', 'policy_id', 'agent_id', 'program_code', 'premium', 'commission_rate', 'program_modifier', 'volume_bonus', 'policy_status', 'status_multiplier', 'adjustments', 'commission_amount', 'payout_date', 'payout_status', 'calc_method', '_bronze_batch_id', 'true_tier_rate', 'true_program_modifier', 'true_volume_bonus', 'true_status_multiplier', 'silver_effective_rate', 'silver_commission_amount', '_silver_load_timestamp', '_silver_batch_id']


In [8]:
from pyspark.sql.functions import sum as spark_sum

reconciliation_df = (
    drift_df
    .withColumn(
        "variance",
        spark_round(col("silver_commission_amount") - col("commission_amount"), 2)
    )
    .withColumn(
        "drift_direction",
        when(col("variance") > 0, "UNDERPAYMENT")
        .when(col("variance") < 0, "OVERPAYMENT")
        .otherwise("NO_DRIFT")
    )
    .withColumn(
        "drift_pattern",
        when(
            col("commission_rate") != col("true_tier_rate"),
            "PATTERN_A"
        )
        .when(
            col("program_modifier") != col("true_program_modifier"),
            "PATTERN_B"
        )
        .when(
            col("volume_bonus") != col("true_volume_bonus"),
            "PATTERN_C"
        )
        .otherwise("UNKNOWN")
    )
    .select(
        "payout_id", "policy_id", "agent_id", "program_code",
        "premium", "payout_date", "payout_status",
        "commission_amount", "silver_commission_amount",
        "variance", "drift_direction", "drift_pattern",
        "commission_rate", "true_tier_rate",
        "program_modifier", "true_program_modifier",
        "volume_bonus", "true_volume_bonus",
        "_bronze_batch_id", "_silver_batch_id"
    )
    .withColumn("_silver_load_timestamp", current_timestamp())
)

# Exclude floating point noise. Any variance < $0.05 is a rounding artifact,
# not a real VBA bug. Real drift patterns produce variances of several dollars minimum.
reconciliation_df = reconciliation_df.filter(
    (col("variance") > 0.05) | (col("variance") < -0.05)
)
row_count = reconciliation_df.count()
print(f"Reconciliation rows: {row_count}")
print()

print("Drift breakdown by pattern and direction:")
reconciliation_df.groupBy("drift_pattern", "drift_direction").count().orderBy("drift_pattern").show()

print("Total variance by pattern (positive = underpayment, negative = overpayment):")
reconciliation_df.groupBy("drift_pattern").agg(
    spark_round(spark_sum("variance"), 2).alias("total_variance")
).orderBy("drift_pattern").show()

StatementMeta(, 488ad106-7cea-435f-956d-2050f45c1443, 10, Finished, Available, Finished, False)

Reconciliation rows: 107

Drift breakdown by pattern and direction:
+-------------+---------------+-----+
|drift_pattern|drift_direction|count|
+-------------+---------------+-----+
|    PATTERN_A|   UNDERPAYMENT|   24|
|    PATTERN_B|   UNDERPAYMENT|   28|
|    PATTERN_B|    OVERPAYMENT|   11|
|    PATTERN_C|    OVERPAYMENT|   44|
+-------------+---------------+-----+

Total variance by pattern (positive = underpayment, negative = overpayment):
+-------------+--------------+
|drift_pattern|total_variance|
+-------------+--------------+
|    PATTERN_A|      13728.19|
|    PATTERN_B|       7757.62|
|    PATTERN_C|      -9165.54|
+-------------+--------------+



In [9]:
(
    reconciliation_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(TARGET_TABLE)
)
print(f"Wrote {reconciliation_df.count()} rows to {TARGET_TABLE}")
print()

# Verify round trip.
verify_df = spark.read.table(TARGET_TABLE)
print(f"Verified: {verify_df.count()} rows, {len(verify_df.columns)} cols")
print()

print("Final summary:")
verify_df.groupBy("drift_pattern", "drift_direction").count().orderBy("drift_pattern").show()

print("Total financial exposure by pattern:")
verify_df.groupBy("drift_pattern").agg(
    spark_round(spark_sum("variance"), 2).alias("total_variance")
).orderBy("drift_pattern").show()

StatementMeta(, 488ad106-7cea-435f-956d-2050f45c1443, 11, Finished, Available, Finished, False)

Wrote 107 rows to silver_commission_reconciliation

Verified: 107 rows, 21 cols

Final summary:
+-------------+---------------+-----+
|drift_pattern|drift_direction|count|
+-------------+---------------+-----+
|    PATTERN_A|   UNDERPAYMENT|   24|
|    PATTERN_B|   UNDERPAYMENT|   28|
|    PATTERN_B|    OVERPAYMENT|   11|
|    PATTERN_C|    OVERPAYMENT|   44|
+-------------+---------------+-----+

Total financial exposure by pattern:
+-------------+--------------+
|drift_pattern|total_variance|
+-------------+--------------+
|    PATTERN_A|      13728.19|
|    PATTERN_B|       7757.62|
|    PATTERN_C|      -9165.54|
+-------------+--------------+

